In [ ]:
## One-off preprocessing: build the oligo-pool FASTA/GTF annotation for the gRNA library
## (defines write_annotation(); not run automatically since it only needs to run once per library)
from pathlib import Path
import pandas as pd


def write_annotation(
    oligo_pool='../data/lipogrid/pilot/sc_RNA/gRNA_oligos/gRNA_oligos.csv',
    output_fasta='../data/lipogrid/pilot/sc_RNA/gRNA_oligos/oligo_pool_plasmid.fa',
    output_gtf='../data/lipogrid/pilot/sc_RNA/gRNA_oligos/oligo_pool_plasmid.gtf',
    output_gtf2='../data/lipogrid/pilot/sc_RNA/gRNA_oligos/oligo_pool_plasmid_structure.gtf',
):
    df = pd.read_csv(oligo_pool, header=None, names=['oligo_name', 'sequence'])
    fasta_entries, gtf_entries, gtf2_entries = [], [], []

    fasta_header_template = ">{chrom}_chrom dna:chromosome chromosome:GRCh38:{chrom}_chrom:1:{length}:1 REF"
    gtf_template = '{chrom}_chrom\thavana\tgene\t1\t{length}\t.\t+\t.\tgene_id "{id}_gene"; gene_name "{id}_gene"; gene_source "ensembl_havana"; gene_biotype "lincRNA";\n'
    gtf2_template = '{chrom}_chrom\thavana\t{type}\t{start}\t{length}\t.\t+\t.\tgene_name "{id}_gene";\n'

    U6_seq = "GAGGGCCTATTTCCCATGATTCCTTCATATTTGCATATACGATACAAGGCTGTTAGAGAGATAATTAGAATTAATTTGACTGTAAACACAAAGATATTAGTACAAAATACGTGACGTAGAAAGTAATAATTTCTTGGGTAGTTTGCAGTTTTAAAATTATGTTTTAAAATGGACTATCATATGCTTACCGTAACTTGAAAGTATTTCGATTTCTTGGCTTTATATATCTTGTGGAAAGGACGAAACACC"
    Rest_seq = "GTTTTAGAGCTAGAAATAGCAAGTTAAAATAAGGCTAGTCCGTTATCAACTTGAAAAAGTGGCACCGAGTCGGTGCTTTTTTAAGCTTGGCGTAACTAGATCTTGAGACACTGCTTTTTGCTTGTACTGGGTCTCTCTGGTTAGACCAGATCTGAGCCTGGGAGCTCTCTGGCTAACTAGGGAACCCACTGCTTAAGCCTCAATAAAGCTTGCCTTGAGTGCTTCAAGTAGTGTGTGCCCGTCTGTTGTGTGACTCTGGT"

    for _, row in df.iterrows():
        oligo_name, guide_sequence = row["oligo_name"], row["sequence"]
        sequence = U6_seq + guide_sequence + Rest_seq
        gRNA_start = len(U6_seq) + 1  # include the additional G as gRNA start
        gRNA_end = len(U6_seq + guide_sequence)
        fasta_entries += [fasta_header_template.format(chrom=oligo_name, length=len(sequence)), sequence]
        gtf_entries.append(gtf_template.format(chrom=oligo_name, id=oligo_name, length=len(sequence)))
        gtf2_entries.append(gtf2_template.format(chrom=oligo_name, type='gene', id=oligo_name, start=1, length=len(sequence)))
        gtf2_entries.append(gtf2_template.format(chrom=oligo_name, type='U6', id=oligo_name, start=1, length=len(U6_seq)))
        gtf2_entries.append(gtf2_template.format(chrom=oligo_name, type='gRNA', id=oligo_name, start=gRNA_start, length=gRNA_end))
        gtf2_entries.append(gtf2_template.format(chrom=oligo_name, type='Rest', id=oligo_name, start=gRNA_end + 1, length=len(sequence)))

    Path(output_fasta).write_text("\n".join(fasta_entries))
    Path(output_gtf).write_text("".join(gtf_entries))
    Path(output_gtf2).write_text("".join(gtf2_entries))


# run once to (re)generate the oligo pool annotation files:
# write_annotation()

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from matplotlib.colors import TwoSlopeNorm, LinearSegmentedColormap
from mpl_toolkits.axes_grid1 import Divider, Size
from adjustText import adjust_text
import seaborn as sns

import scanpy as sc
import anndata as ad

from sklearn.cluster import AgglomerativeClustering
from scipy.cluster.hierarchy import linkage, leaves_list
from scipy.stats import mannwhitneyu, gaussian_kde
from statsmodels.stats.multitest import multipletests

sys.path.append(str(Path.cwd().parent / "scripts"))
from style import set_default_style, set_publication_style
from grna_assignment import assign_grna_to_cell

set_default_style()

In [ ]:
## first inspection of raw data
count_matrix_r1 = sc.read_10x_h5('../data/lipogrid/pilot/sc_RNA/mapping/extra_seq/Chemv3_lipogrid_CROP_extraseq/outs/filtered_feature_bc_matrix.h5')
count_matrix_r2 = sc.read_10x_h5('../data/lipogrid/pilot/sc_RNA/mapping/second_CROP_251211/Chemv3_lipogrid_CROP_2nd_251211/outs/filtered_feature_bc_matrix.h5')

## add run info to anndata objects
count_matrix_r1.obs['run'] = 'run1'
count_matrix_r2.obs['run'] = 'run2'
count_matrix_r2.obs.index = count_matrix_r2.obs.index.str.replace('-1', '-2')
## merge both runs
count_matrix_r1.var_names_make_unique()
count_matrix_r2.var_names_make_unique()
count_matrix = ad.concat([count_matrix_r1, count_matrix_r2], axis=0, join="outer")
count_matrix.var = count_matrix_r1.var.copy()
# remove var names with _gene from count matrix
count_matrix = count_matrix[:, ~count_matrix.var_names.str.contains('_gene')].copy()
count_matrix

In [ ]:
## calculate general statistics on data
sc.pp.calculate_qc_metrics(count_matrix, inplace=True)
count_matrix

## Determined which gRNAs are expressed in each cell and perform gRNA assignmentg the similar way as in 10x Xenium


In [ ]:
## gRNAs are a seperate library, so load and process separately then link with RNA data
count_matrix_gRNA_r1 = sc.read_10x_h5('../data/lipogrid/pilot/sc_RNA/mapping/Chemv3_lipogrid_CROP_gRNA_extraseq/outs/filtered_feature_bc_matrix.h5')
count_matrix_gRNA_r2 = sc.read_10x_h5('../data/lipogrid/pilot/sc_RNA/mapping/second_CROP_251211/Chemv3_lipogrid_CROP_gRNA_2nd_251211/outs/filtered_feature_bc_matrix.h5')
## add run info to anndata objects
count_matrix_gRNA_r1.obs['run'] = 'run1'
count_matrix_gRNA_r2.obs['run'] = 'run2'
count_matrix_gRNA_r2.obs.index = count_matrix_gRNA_r2.obs.index.str.replace('-1', '-2')
## merge both runs
count_matrix_gRNA_r1.var_names_make_unique()
count_matrix_gRNA_r2.var_names_make_unique()
count_matrix_gRNA = ad.concat([count_matrix_gRNA_r1, count_matrix_gRNA_r2], axis=0, join="outer")
count_matrix_gRNA.var = count_matrix_gRNA_r1.var.copy()
# remove var names with _gene from count matrix
count_matrix_gRNA_filtered = count_matrix_gRNA[:, count_matrix_gRNA.var_names.str.contains('_gene')].copy()
count_matrix_gRNA_filtered

In [ ]:
if hasattr(count_matrix_gRNA_filtered.X, "toarray"):
    dense_X = count_matrix_gRNA_filtered.X.toarray()
else:
    dense_X = count_matrix_gRNA_filtered.X

count_matrix_gRNA_filtered_df = pd.DataFrame(
    dense_X,
    index=count_matrix_gRNA_filtered.obs.index,
    columns=count_matrix_gRNA_filtered.var.index
)
count_matrix_gRNA_filtered_df

In [ ]:
cell_gRNA = assign_grna_to_cell(count_matrix_gRNA_filtered_df, min_count=10, multi_ratio=1.4, min_fraction=0.20)
cell_gRNA

In [ ]:
count_matrix.obs['assigned_gRNA'] = count_matrix.obs.index.map(cell_gRNA)
count_matrix.obs['assigned_gRNA'].value_counts()

In [ ]:
# explore data for filtering
fig, axs = plt.subplots(1, 3, figsize=(10, 3.5))

axs[0].hist(count_matrix.obs['log1p_total_counts'], bins=40, color='blue', alpha=1, edgecolor='black')
axs[0].set_title('log1p_total_counts')
axs[0].set_xlabel('log1p_total_counts')
axs[0].set_ylabel('# cells')
# add vertical lines at 8 and 10.5
axs[0].axvline(x=8.4, color='red', linestyle='--')
axs[0].axvline(x=10.7, color='red', linestyle='--')

axs[1].hist(np.log2(count_matrix.var['n_cells_by_counts']+1), bins=40, color='blue', alpha=1, edgecolor='black')

axs[1].set_title('n_cells_by_counts')
axs[1].set_xlabel('n_cells_by_counts')
axs[1].set_ylabel('# genes')
# add vertical line at 2
axs[1].axvline(x=2, color='red', linestyle='--')

## get to a average count per cell
count_per_cell = ((count_matrix.var['total_counts']+1)/(count_matrix.var['n_cells_by_counts']+1))
count_per_cell = count_per_cell.clip(upper=1.5)

axs[2].hist(count_per_cell, bins=40, color='blue', alpha=1, edgecolor='black')

axs[2].set_title('total_counts/n_cells_by_counts')
axs[2].set_xlabel('total_counts/n_cells_by_counts')
axs[2].set_ylabel('# genes')
axs[2].axvline(x=1.05, color='red', linestyle='--')

plt.tight_layout()

In [ ]:
## filtering on cells and genes
sc.pp.calculate_qc_metrics(count_matrix, inplace=True)
# filter cells
count_matrix_filtered = count_matrix[count_matrix.obs['log1p_total_counts']>8.4,:].copy()  
count_matrix_filtered = count_matrix_filtered[count_matrix_filtered.obs['log1p_total_counts']<10.7,:].copy()

## filter genes 
count_matrix_filtered = count_matrix_filtered[:,(count_matrix_filtered.var['total_counts']+1)/(count_matrix_filtered.var['n_cells_by_counts']+1)>1.1].copy()
count_matrix_filtered.write_h5ad("../data/lipogrid/pilot/sc_RNA/analyses/count_matrix_filtered.h5ad")
count_matrix_filtered

In [ ]:
sc.pp.neighbors(count_matrix_filtered, n_neighbors=10)
sc.tl.umap(count_matrix_filtered, min_dist=0.5, spread=1.0, random_state=42)
sc.pl.umap(count_matrix_filtered, size=18, wspace=0, legend_fontsize=15, legend_fontweight='bold', color='run', show=True)

In [ ]:
sc.pp.normalize_total(count_matrix_filtered)
sc.tl.pca(count_matrix_filtered)
sc.pl.pca_variance_ratio(count_matrix_filtered, n_pcs=50, log=True)

In [ ]:
# Only keep cells with a gRNA assigned _gene
# remove cells without gRNA assigned
count_matrix_filtered_g = count_matrix_filtered[~count_matrix_filtered.obs['assigned_gRNA'].isna(), :].copy()
# remove cells with multiple or low gRNAs assigned
count_matrix_filtered_g = count_matrix_filtered_g[count_matrix_filtered_g.obs['assigned_gRNA'].str.contains('_gene'), :].copy()
# create extra column with target gene only
count_matrix_filtered_g.obs['target_gene'] = count_matrix_filtered_g.obs['assigned_gRNA'].str.split('.').str[0]
count_matrix_filtered_g

In [ ]:
# Count cells per gRNA per sample_id
gRNA_sample_counts = count_matrix_filtered_g.obs['target_gene'].value_counts()

# remove control / non-targeting categories
exclude = ['low_count', 'multiple_gRNAs', 'ambiguous', 'Intergenic', 'no_gRNA']
gRNA_sample_counts_filtered = gRNA_sample_counts.drop(index=exclude, errors='ignore')

# Order by total count (descending)
gRNA_order = gRNA_sample_counts_filtered.sort_values(ascending=False).index
gRNA_sample_counts_ordered = gRNA_sample_counts_filtered.loc[gRNA_order]

# Plot
set_publication_style()
fig, ax = plt.subplots(figsize=(12, 6), dpi=150)
ax.bar(gRNA_sample_counts_ordered.index, gRNA_sample_counts_ordered.values,
       color="#601fb4", width=0.8, edgecolor="none")

ax.set_xlabel('gRNA target gene')
ax.set_ylabel('Number of cells assigned')
ax.tick_params(axis='y', length=5, width=1)
ax.tick_params(axis='x', length=0)
ax.set_xticks([])   # many genes: hide x-tick labels (as in your original)
ax.margins(x=0) 
fig.tight_layout()
fig.savefig(
    "../data/lipogrid/pilot/analysis/internalnorm_finalfigs/barplot_gRNA_percell_perGene_CROPseq.pdf",
    dpi=300, bbox_inches="tight",
)
plt.show()

In [ ]:
## save final filtered gRNA anndata object
count_matrix_filtered_g.write('../data/lipogrid/pilot/sc_RNA/analyses/count_matrix_filtered_g.h5ad')


In [ ]:
if hasattr(count_matrix_filtered_g.X, "toarray"):
    dense_X = count_matrix_filtered_g.X.toarray()
else:
    dense_X = count_matrix_filtered_g.X

count_matrix_filtered_g_df = pd.DataFrame(
    dense_X,
    index=count_matrix_filtered_g.obs['target_gene'],
    columns=count_matrix_filtered_g.var.index
)
count_matrix_filtered_g_df

In [ ]:
## per-gene KO effect on RNA expression vs. intergenic controls (log2FC + Mann-Whitney p-value)
RESULTS_CSV = '../data/lipogrid/pilot/analysis/CROP_seq_log2FC_manw_pergeneKO_final.csv'

if not Path(RESULTS_CSV).exists():
    all_results_df = pd.DataFrame(index=count_matrix_filtered_g_df.columns)

    intergenic_mask = count_matrix_filtered_g_df.index.str.contains('Intergenic')
    intergenic_gRNAs = count_matrix_filtered_g_df[intergenic_mask].values

    df_values = count_matrix_filtered_g_df.values
    df_index = count_matrix_filtered_g_df.index.to_numpy()
    unique_targets = np.unique(df_index)

    for target in unique_targets:
        target_gRNAs = df_values[df_index == target]

        target_gRNAs_mean = target_gRNAs.mean(axis=0)
        intergenic_gRNAs_mean = intergenic_gRNAs.mean(axis=0)
        target_gRNAs_lfc = np.log2((target_gRNAs_mean + 1) / (intergenic_gRNAs_mean + 1))

        p_values = [
            mannwhitneyu(target_gRNAs[:, i], intergenic_gRNAs[:, i], alternative='two-sided')[1]
            for i in range(target_gRNAs.shape[1])
        ]

        all_results_df[f'{target}_log2FC'] = target_gRNAs_lfc
        all_results_df[f'{target}_pvalue'] = p_values

    all_results_df.to_csv(RESULTS_CSV)

all_results_df = pd.read_csv(RESULTS_CSV, index_col=0)
all_results_df

In [ ]:
# For each column ending with '_pvalue', perform FDR correction and insert a new column with '_FDR' immediately after
for col in list(all_results_df.columns):
    if col.endswith('_pvalue'):
        pvals = all_results_df[col].values
        _, fdr, _, _ = multipletests(pvals, method='fdr_bh')
        fdr_col = col.replace('_pvalue', '_FDR')
        # Insert FDR column immediately after pvalue column
        col_idx = all_results_df.columns.get_loc(col)
        all_results_df.insert(col_idx + 1, fdr_col, fdr)

In [ ]:
# Filter out gRNAs to proceed with same gRNAs as in LipoGrid
with open('../data/lipogrid/pilot/analysis/final_4_runs/filtered_143target_genes.txt', 'r') as f:
    filtered_143target_genes = [line.strip() for line in f]

# keep columns in all_results_df if the name before _log2FC, _pvalue or _FDR is in filtered_143target_genes
filtered_columns = []
for col in all_results_df.columns:
    base_name = col.rsplit('_', 1)[0]  # Get the part before the last underscore
    if base_name in filtered_143target_genes:
        filtered_columns.append(col)

# subset all_results_df to only include filtered_columns
filtered_all_results_df = all_results_df[filtered_columns]
filtered_all_results_df.to_csv('../data/lipogrid/pilot/analysis/CROP_seq_log2FC_manw_pergeneKO_sameKOs.csv')

filtered_all_results_df

In [ ]:
# keep rows if at least one FDR value is below 0.05 for comprehensive heatmap
fdr_columns = [col for col in filtered_all_results_df.columns if col.endswith('_FDR')]
mask = (filtered_all_results_df[fdr_columns] < 0.05).any(axis=1)
all_results_df_FDR = filtered_all_results_df[mask]
all_results_df_FDR

In [ ]:
# Select log2FC columns
log2fc_columns = [col for col in all_results_df_FDR.columns if col.endswith('_log2FC')]
log2fc_matrix = all_results_df_FDR[log2fc_columns]
log2fc_matrix.columns = [col.replace('_log2FC', '') for col in log2fc_matrix.columns]
log2fc_matrix = log2fc_matrix.T
## clip data to -1 and 1 for better visualization
log2fc_matrix = log2fc_matrix.clip(lower=-0.5, upper=0.5)
# Set the number of clusters
n_clusters_cols = 8
# Fit AgglomerativeClustering to the transposed data (features as columns)
agglo = AgglomerativeClustering(n_clusters=n_clusters_cols, linkage='ward')
agglo_labels = agglo.fit_predict(log2fc_matrix.T)

n_clusters_rows = 5
agglo_rows = AgglomerativeClustering(n_clusters=n_clusters_rows, linkage='ward')
agglo_labels_rows = agglo_rows.fit_predict(log2fc_matrix)

# Assign a color to each cluster

unique_clusters = np.unique(agglo_labels)
palette = sns.color_palette("Set2", len(unique_clusters))
cluster_colors = dict(zip(unique_clusters, palette))

unique_clusters_row = np.unique(agglo_labels_rows)
palette_row = sns.color_palette("Set2", len(unique_clusters_row))
cluster_colors_row = dict(zip(unique_clusters_row, palette_row))

# Map each column to its cluster color
col_colors = pd.Series(agglo_labels, index=log2fc_matrix.columns).map(cluster_colors)
row_colors = pd.Series(agglo_labels_rows, index=log2fc_matrix.index).map(cluster_colors_row)

# MANUAL CLUSTER ORDER
manual_order_col = [5,2,1,4,7,3,0,6]  # <-- change this to your desired order
manual_order_row = [4,2,0,1,3]  # <-- change this to your desired order

# Get column indices in the manual order
ordered_cols = []
for cl in manual_order_col:
    ordered_cols.extend(log2fc_matrix.columns[agglo_labels == cl])

# Get row indices in the manual order
ordered_rows = []
for cl in manual_order_row:
    ordered_rows.extend(log2fc_matrix.index[agglo_labels_rows == cl])

# Hierarchical clustering within each column cluster
ordered_cols_hier = []
for cl in manual_order_col:
    cols_in_cl = log2fc_matrix.columns[agglo_labels == cl]
    if len(cols_in_cl) > 1:
        # Hierarchical clustering for columns in this cluster
        Z = linkage(log2fc_matrix[cols_in_cl].T, method='complete') # average
        idx = leaves_list(Z)
        ordered = cols_in_cl[idx]
    else:
        ordered = cols_in_cl
    ordered_cols_hier.extend(ordered)

# Hierarchical clustering within each row cluster
ordered_rows_hier = []
for cl in manual_order_row:
    rows_in_cl = log2fc_matrix.index[agglo_labels_rows == cl]
    if len(rows_in_cl) > 1:
        # Hierarchical clustering for rows in this cluster
        Z = linkage(log2fc_matrix.loc[rows_in_cl], method='complete') # average
        idx = leaves_list(Z)
        ordered = rows_in_cl[idx]
    else:
        ordered = rows_in_cl
    ordered_rows_hier.extend(ordered)

# Use previous row ordering, or apply similar clustering for rows if needed
adata_norm_grouped_lfc_orderd_hier = log2fc_matrix.loc[ordered_rows_hier, ordered_cols]

g = sns.clustermap(
    adata_norm_grouped_lfc_orderd_hier,
    cmap='RdYlBu_r',
    figsize=(22, 10),
    col_cluster=False,
    row_cluster=False,
    cbar_kws={'label': 'log2FC'},
    xticklabels=False,
    yticklabels=False,
    col_colors=col_colors[ordered_cols_hier],
    row_colors=row_colors[ordered_rows]
)
g.ax_row_dendrogram.set_visible(False)
g.cax.set_position([.08, .3, .018, .25])
plt.show()

In [ ]:
## save the gene <-> cluster assignments (consumed by GSEA_CROP_seq.ipynb)
CLUSTER_DIR = Path('../data/lipogrid/pilot/analysis/CROP_seq')

target_gene_clusters = pd.DataFrame({
    'gene': adata_norm_grouped_lfc_orderd_hier.index,
    'cluster': [agglo_labels_rows[log2fc_matrix.index.get_loc(gene)] for gene in adata_norm_grouped_lfc_orderd_hier.index],
})
gene_clustered = pd.DataFrame({
    'gene': adata_norm_grouped_lfc_orderd_hier.columns,
    'cluster': [agglo_labels[log2fc_matrix.columns.get_loc(gene)] for gene in adata_norm_grouped_lfc_orderd_hier.columns],
})

target_gene_clusters.to_csv(CLUSTER_DIR / 'target_genes_CROPseq-clusters.tsv', sep='\t', index=False)
gene_clustered.to_csv(CLUSTER_DIR / 'genes_CROPseq_10cols_clusters.tsv', sep='\t', index=False)

In [ ]:
# Plot heatmap with columns ordered by manual cluster order
g = sns.clustermap(
    adata_norm_grouped_lfc_orderd_hier,
    cmap='RdYlBu_r',
    figsize=(32, 32),
    col_cluster=False,          # No hierarchical clustering
    row_cluster=False,
    cbar_kws={'label': 'log2FC'},
    xticklabels=False,
    yticklabels=True,
    col_colors=col_colors[ordered_cols],
    row_colors=row_colors[ordered_rows],
    rasterized=True,            # keeps the heatmap raster; text/labels stay vector
)
g.ax_row_dendrogram.set_visible(False)
g.ax_heatmap.set_yticklabels(g.ax_heatmap.get_yticklabels(), fontsize=14)
g.cax.set_position([.13, .3, .018, .25])  # [left, bottom, width, height] in figure coords

# style the colorbar to match (1px, editable label)
g.cax.set_ylabel('log2FC', fontsize=30, fontweight='bold')
g.cax.tick_params(length=5, width=1, labelsize=30)

# Add cluster numbers above columns
for cl_num in manual_order_col:
    col_inds = [i for i, col in enumerate(ordered_cols)
                if agglo_labels[log2fc_matrix.columns.get_loc(col)] == cl_num]
    if col_inds:
        center = np.mean(col_inds)
        g.ax_heatmap.text(center, -2, str(cl_num), ha='center', va='center',
                          fontsize=28, fontweight='bold', color='black')

# Add cluster numbers beside rows
for cl_num in manual_order_row:
    row_inds = [i for i, row in enumerate(ordered_rows)
                if agglo_labels_rows[log2fc_matrix.index.get_loc(row)] == cl_num]
    if row_inds:
        center = np.mean(row_inds)
        g.ax_heatmap.text(-2, center, str(cl_num), ha='right', va='top',
                          fontsize=28, fontweight='bold', color='black', rotation=0)

# save as pdf
g.savefig(
    '../data/lipogrid/pilot/analysis/CROP_seq/heatmap_log2FC_CROP_seq_FDR0.05.final.pdf',
    dpi=300, bbox_inches="tight",
)
plt.show()

In [ ]:
## Volcano plot for NPC1, colored by functional gene category, non-sig points shaded by local density
def _register_font(font_path):
    """Register a font file with matplotlib and return its family name."""
    if not font_path:
        return None
    fm.fontManager.addfont(font_path)
    return fm.FontProperties(fname=font_path).get_name()


# parameters
target = 'NPC1'
font_path = '../data/fonts/HelveticaNeue_ttf/HelveticaNeue.ttf'
pval_cap = 1e-8
sig_pval = 0.01
sig_lfc = 0.25
box_px = 300
dpi = 100
dot_size = 12
save_dir = '../data/lipogrid/pilot/analysis/internalnorm_finalfigs'

# grey85 -> black density ramp, matching densCols(colramp=colorRampPalette(c("grey85","black")))
dens_cmap = LinearSegmentedColormap.from_list("densgrey", ["#D9D9D9", "#000000"])

highlight_genes = ['SQLE', 'HMGCS1', 'FDPS', 'MSMO1', 'FDFT1', 'ACAT2', 'IDI1', 'EBP', 'MVD']
transcription_factor = ['SREBF2']
cholesterol_involved = ['ERG28', 'INSIG1', 'LRP8']


def gene_color(gene):
    if gene in highlight_genes:
        return 'purple'
    if gene in transcription_factor:
        return 'green'
    if gene in cholesterol_involved:
        return 'darkorange'
    return 'cornflowerblue'


# cap lowest p-value on a copy so the original is untouched
df = all_results_df.copy()
df[f'{target}_pvalue'] = df[f'{target}_pvalue'].apply(lambda x: max(x, pval_cap))

# register Helvetica Neue if available, else fall back
registered = _register_font(font_path)
sans_list = ([registered] if registered else []) + \
    ['Helvetica Neue', 'Helvetica', 'Arial', 'DejaVu Sans']

# publication-ready figure style
plt.rcParams.update({
    'font.family': 'sans-serif',
    'font.sans-serif': sans_list,
    'font.weight': 'normal',
    'font.size': 8,
    'axes.titlesize': 12,
    'axes.titleweight': 'normal',
    'axes.labelsize': 12,
    'axes.labelweight': 'normal',
    'xtick.labelsize': 8,
    'ytick.labelsize': 8,
    'axes.linewidth': 1,
    'lines.linewidth': 1,
    'grid.linewidth': 1,
    'pdf.fonttype': 42,
})

# fixed box_px square plot box
box_in = box_px / dpi
fig = plt.figure(figsize=(box_in + 2.5, box_in + 2.0), dpi=dpi)
h = [Size.Fixed(1.5), Size.Fixed(box_in)]
v = [Size.Fixed(1.0), Size.Fixed(box_in)]
divider = Divider(fig, (0, 0, 1, 1), h, v, aspect=False)
ax = fig.add_axes(divider.get_position(), axes_locator=divider.new_locator(nx=1, ny=1))

# masks
x = df[f'{target}_log2FC']
y = -np.log10(df[f'{target}_pvalue'])
sig_mask = (df[f'{target}_pvalue'] < sig_pval) & (df[f'{target}_log2FC'].abs() > sig_lfc)

# non-significant points: colored by local 2D density (densCols-style)
ns = ~sig_mask
xns = x[ns].to_numpy()
yns = y[ns].to_numpy()
ok = np.isfinite(xns) & np.isfinite(yns)
xns, yns = xns[ok], yns[ok]

if len(xns) > 2:
    dens = gaussian_kde(np.vstack([xns, yns]))(np.vstack([xns, yns]))
    order = dens.argsort()                       # densest plotted last (on top)
    ax.scatter(xns[order], yns[order], c=dens[order], cmap=dens_cmap,
               s=dot_size, edgecolor='none', rasterized=True, zorder=1)
else:
    ax.scatter(xns, yns, c='#D9D9D9', s=dot_size, zorder=1)

# significant points in category color
sig_colors = [gene_color(g) for g in df.index[sig_mask]]
ax.scatter(x[sig_mask], y[sig_mask], c=sig_colors, s=dot_size, rasterized=False, zorder=2)

texts = []
for lipid, row in df[sig_mask].iterrows():
    texts.append(
        ax.text(row[f'{target}_log2FC'], -np.log10(row[f'{target}_pvalue']), lipid,
                fontsize=8, color=gene_color(lipid), ha='center', va='center')
    )

ax.axvline(x=0, color='grey', linestyle='--', linewidth=1, label='Fold Change = 1')

ax.set_title(f'Effect {target} knockout')
ax.set_xlabel(f'Log$_2$({target} gRNA/intergenic gRNA)')
ax.set_ylabel('-Log$_{10}$(p-value)')
ax.grid(False)
ax.tick_params(length=5, width=1)

ymax = float(np.nanmax(-np.log10(df[f'{target}_pvalue'])))
ax.set_ylim(-0.05, ymax * 1.05)
ax.margins(y=0)

adjust_text(
    texts,
    arrowprops=dict(arrowstyle='-', color='gray', lw=1),
    expand_text=(1.3, 1.3),
    expand_points=(1.2, 1.2),
    force_text=0.3,
    force_points=0.3,
    force_pull=7,
    time_lim=10,
    min_arrow_len=15,
)

fig.savefig(
    f'{save_dir}/volcano_plot_{target}_log2FC{sig_lfc}_p{sig_pval}_colored_nocenter_CROP_seq.pdf',
    dpi=300, bbox_inches='tight',
)
plt.show()

In [ ]:
# cholesterol and fatty-acid importers, filtered down to genes that pass the expression cutoff (mean_counts >= 1)
chol_importers = ['LDLR', 'VLDLR', 'LRP1', 'LRP2', 'LRP4', 'LRP5', 'LRP6', 'LRP8',
                  'SCARB1', 'SCARB2', 'CD36', 'CXCL16', 'OLR1', 'MSR1',
                  'NPC1L1', 'SORT1', 'SORL1', 'LRPAP1',
                  'APOE', 'APOB', 'APOA1', 'APOA4',
                  'LPL', 'LIPC', 'LIPG', 'LIPA']
fa_importers = ['CD36', 'SLC27A1', 'SLC27A2', 'SLC27A3', 'SLC27A4', 'SLC27A5', 'SLC27A6',
                'FABP1', 'FABP2', 'FABP3', 'FABP4', 'FABP5', 'FABP6', 'FABP7', 
                'CAV1', 'CAV2']

expressed = set(count_matrix_filtered.var_names[count_matrix_filtered.var['mean_counts'] >= 1])

def keep_expressed(genes):
    return [g for g in genes if g in expressed]

chol_importers_f = keep_expressed(chol_importers)
fa_importers_f   = keep_expressed(fa_importers)

# optional: see what was removed and why (low vs not measured at all)
measured = set(count_matrix_filtered.var_names)
for name, lst in [('cholesterol', chol_importers), ('fatty-acid', fa_importers)]:
    low        = [g for g in lst if g in measured and g not in expressed]   # mean_counts < 1
    not_meas   = [g for g in lst if g not in measured]                      # absent from matrix
    print(f"{name}: kept {keep_expressed(lst)}")
    print(f"   dropped (mean_counts<1): {low}")
    print(f"   dropped (not measured):  {not_meas}\n")

In [ ]:
set_publication_style()

# data
df = all_results_df.copy()
chol_import = keep_expressed(chol_importers)
fa_import   = keep_expressed(fa_importers)
rows = [r for r in chol_import + fa_import if r in df.index]

chol_ko = ['HMGCR','HMGCS1','FDFT1','DHCR7']
fa_ko   = ['FASN','HSD17B12','SCD','ELOVL6']
kos = chol_ko + fa_ko
FC = pd.DataFrame({k: df.loc[rows, f'{k}_log2FC'].astype(float) for k in kos}, index=rows).fillna(0)
P  = pd.DataFrame({k: df.loc[rows, f'{k}_pvalue'].astype(float) for k in kos}, index=rows)

# plot
fig, ax = plt.subplots(figsize=(0.62*len(kos)+3.5, 0.52*len(rows)+3), dpi=150)
vmax = max(np.nanpercentile(np.abs(FC.values), 98), 0.25)
im = ax.imshow(FC.values, cmap='RdBu_r',
               norm=TwoSlopeNorm(vmin=-vmax, vcenter=0, vmax=vmax), aspect='auto')

ax.set_xticks(range(len(kos))); ax.set_xticklabels(kos, rotation=45, ha='right')
ax.set_yticks(range(len(rows))); ax.set_yticklabels(rows)
ax.tick_params(length=0)                       # clean: no tick marks on the heatmap
for s in ax.spines.values():                   # thin 1px frame
    s.set_linewidth(1)

for i in range(len(rows)):
    for j, k in enumerate(kos):
        v, p = FC.iloc[i, j], P.iloc[i, j]
        sig = pd.notna(p) and p < 0.05
        ax.text(j, i, f"{v:+.2f}" + ("*" if sig else ""), ha='center', va='center',
                fontsize=9, fontweight='bold' if sig else 'normal',
                color='white' if abs(v) > vmax*0.6 else 'black')

nci, nck = len(chol_import), len(chol_ko)
ax.axhline(nci-0.5, color='k', lw=1); ax.axvline(nck-0.5, color='k', lw=1)
ax.text(-0.09, (nci-1)/2, 'Cholesterol /\nlipoprotein\nimport', transform=ax.get_yaxis_transform(),
        ha='right', va='center', fontsize=12, fontweight='bold', color='#601fb4')
ax.text(-0.09, nci+(len(rows)-nci-1)/2, 'Fatty-acid\nimport', transform=ax.get_yaxis_transform(),
        ha='right', va='center', fontsize=12, fontweight='bold', color='#28b2b4')
ax.text((nck-1)/2, 1.03, 'Cholesterol synthesis KOs', transform=ax.get_xaxis_transform(),
        ha='center', va='bottom', fontsize=13, fontweight='bold', color='#601fb4')
ax.text(nck+(len(kos)-nck-1)/2, 1.03, 'Fatty-acid synthesis KOs', transform=ax.get_xaxis_transform(),
        ha='center', va='bottom', fontsize=13, fontweight='bold', color='#28b2b4')

cb = fig.colorbar(im, ax=ax, fraction=0.035, pad=0.02)
cb.set_label('expression log2FC (KO vs control)', fontweight='bold')
cb.outline.set_linewidth(1); cb.ax.tick_params(length=5, width=1)

ax.set_title('Lipid-importer expression under core cholesterol & FA biosynthesis knockouts\n'
             '(values = log2FC; * p < 0.05; CROP-seq)', pad=40)

fig.tight_layout()
fig.savefig(
    "../data/lipogrid/pilot/analysis/internalnorm_finalfigs/lipid_importer_heatmap.pdf",
    dpi=300, bbox_inches="tight",
)
plt.show()